# 03 - Análisis exploratorio de datos

**Proyecto:** Analítica de ventas de videojuegos en Databricks  
**Autor:** Darren J. Blackman M.  
**Asignatura:** Análisis de Datos y Toma de Decisiones en Computación

## Objetivo
Describir la distribución de los datos, identificar tendencias, comparar categorías y detectar valores atípicos mediante PySpark.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

CATALOGO = spark.sql("SELECT current_catalog()").first()[0]
ESQUEMA = "default"

def tabla(nombre):
    return f"`{CATALOGO}`.`{ESQUEMA}`.`{nombre}`"

print(f"Catálogo activo: {CATALOGO}")
print(f"Esquema utilizado: {ESQUEMA}")

Catálogo activo: workspace
Esquema utilizado: default


In [0]:
df = spark.table(tabla("vgsales_limpio"))
print(f"Registros analizados: {df.count():,}")
display(df.limit(5))

Registros analizados: 16,327


Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Decada,Region_Dominante
7,New Super Mario Bros.,DS,2006,Platform,Nintendo,11.38,9.23,6.5,2.9,30.01,2000,Norteamérica
13,Pokemon Gold/Pokemon Silver,GB,1999,Role-Playing,Nintendo,9.0,6.18,7.2,0.71,23.1,1990,Norteamérica
18,Grand Theft Auto: San Andreas,PS2,2004,Action,Take-Two Interactive,9.43,0.4,0.41,10.57,20.81,2000,Otras regiones
20,Brain Age: Train Your Brain in Minutes a Day,DS,2005,Misc,Nintendo,4.75,9.26,4.16,2.05,20.22,2000,Europa
46,Pokemon HeartGold/Pokemon SoulSilver,DS,2009,Action,Nintendo,4.4,2.77,3.96,0.77,11.9,2000,Norteamérica


In [0]:
# Estadísticas descriptivas de las variables de ventas
ventas_cols = ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales"]
display(df.select(ventas_cols).describe())

summary,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
count,16327,16327,16327,16327,16327
mean,0.265414956819999,0.14755435781221332,0.07866111349298374,0.04832547314264308,0.5402315183438703
stddev,0.8215908503623849,0.5087656986961213,0.31155698159943107,0.18988542720450996,1.565731889462265
min,0.0,0.0,0.0,0.0,0.01
max,41.49,29.02,10.22,10.57,82.74


In [0]:
# Cuartiles y mediana de las ventas globales
q1, mediana, q3 = df.approxQuantile("Global_Sales", [0.25, 0.50, 0.75], 0.001)
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr

print(f"Q1: {q1:.3f}")
print(f"Mediana: {mediana:.3f}")
print(f"Q3: {q3:.3f}")
print(f"Límite superior IQR: {limite_superior:.3f}")
print(f"Posibles valores atípicos: {df.filter(F.col('Global_Sales') > limite_superior).count():,}")

Q1: 0.060
Mediana: 0.170
Q3: 0.470
Límite superior IQR: 1.085
Posibles valores atípicos: 1,873


**Interpretación:** la distribución de ventas globales suele presentar asimetría positiva: la mayoría de los videojuegos registra ventas moderadas o bajas, mientras que un grupo pequeño concentra valores muy altos.

In [0]:
# Top 10 videojuegos
top_juegos = (
    df.select("Name", "Platform", "Year", "Genre", "Global_Sales")
      .orderBy(F.desc("Global_Sales"))
      .limit(10)
)
display(top_juegos)

Name,Platform,Year,Genre,Global_Sales
Wii Sports,Wii,2006,Sports,82.74
Super Mario Bros.,NES,1985,Platform,40.24
Mario Kart Wii,Wii,2008,Racing,35.82
Wii Sports Resort,Wii,2009,Sports,33.0
Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,31.37
Tetris,GB,1989,Puzzle,30.26
New Super Mario Bros.,DS,2006,Platform,30.01
Wii Play,Wii,2006,Misc,29.02
New Super Mario Bros. Wii,Wii,2009,Platform,28.62
Duck Hunt,NES,1984,Shooter,28.31


In [0]:
# Ventas por plataforma
ventas_plataforma = (
    df.groupBy("Platform")
      .agg(
          F.count("*").alias("Cantidad_Juegos"),
          F.round(F.sum("Global_Sales"), 2).alias("Ventas_Globales"),
          F.round(F.avg("Global_Sales"), 3).alias("Promedio_por_Juego")
      )
      .orderBy(F.desc("Ventas_Globales"))
)
display(ventas_plataforma.limit(15))

Platform,Cantidad_Juegos,Ventas_Globales,Promedio_por_Juego
PS2,2127,1233.46,0.58
X360,1235,969.61,0.785
PS3,1304,949.35,0.728
Wii,1290,909.81,0.705
DS,2133,818.96,0.384
PS,1189,727.39,0.612
GBA,811,313.56,0.387
PSP,1197,291.71,0.244
PS4,336,278.1,0.828
PC,943,255.05,0.27


**Interpretación:** una plataforma puede liderar por ventas acumuladas debido a su catálogo y permanencia en el mercado. El promedio por juego ayuda a distinguir volumen de catálogo frente a rendimiento individual.

In [0]:
# Ventas por género
ventas_genero = (
    df.groupBy("Genre")
      .agg(
          F.count("*").alias("Cantidad_Juegos"),
          F.round(F.sum("Global_Sales"), 2).alias("Ventas_Globales"),
          F.round(F.avg("Global_Sales"), 3).alias("Promedio_por_Juego")
      )
      .orderBy(F.desc("Ventas_Globales"))
)
display(ventas_genero)

Genre,Cantidad_Juegos,Ventas_Globales,Promedio_por_Juego
Action,3253,1722.88,0.53
Sports,2304,1309.24,0.568
Shooter,1282,1026.2,0.8
Role-Playing,1471,923.84,0.628
Platform,876,829.15,0.947
Misc,1710,797.62,0.466
Racing,1226,726.77,0.593
Fighting,836,444.05,0.531
Simulation,851,390.16,0.458
Puzzle,571,242.22,0.424


In [0]:
# Evolución anual
ventas_anio = (
    df.groupBy("Year")
      .agg(F.round(F.sum("Global_Sales"), 2).alias("Ventas_Globales"))
      .orderBy("Year")
)
display(ventas_anio)

Year,Ventas_Globales
1980,11.38
1981,35.77
1982,28.86
1983,16.79
1984,50.36
1985,53.94
1986,37.07
1987,21.74
1988,47.22
1989,73.45


Databricks visualization. Run in Databricks to view.

In [0]:
# Comparación regional
ventas_region = spark.createDataFrame([
    ("Norteamérica", df.agg(F.sum("NA_Sales")).first()[0]),
    ("Europa", df.agg(F.sum("EU_Sales")).first()[0]),
    ("Japón", df.agg(F.sum("JP_Sales")).first()[0]),
    ("Otras regiones", df.agg(F.sum("Other_Sales")).first()[0]),
], ["Region", "Ventas"])
display(ventas_region.orderBy(F.desc("Ventas")))

Region,Ventas
Norteamérica,4333.430000000124
Europa,2409.1200000000067
Japón,1284.2999999999456
Otras regiones,789.0099999999336


In [0]:
# Correlaciones con ventas globales
correlaciones = []
for c in ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"]:
    correlaciones.append((c, float(df.stat.corr(c, "Global_Sales"))))

display(spark.createDataFrame(correlaciones, ["Variable", "Correlacion_Global"]))

Variable,Correlacion_Global
NA_Sales,0.9412676602364911
EU_Sales,0.9032709811520526
JP_Sales,0.6127937679014456
Other_Sales,0.7479741951455418


In [0]:
# Top 10 publicadoras por ventas
ventas_publicadora = (
    df.groupBy("Publisher")
      .agg(
          F.count("*").alias("Cantidad_Juegos"),
          F.round(F.sum("Global_Sales"), 2).alias("Ventas_Globales")
      )
      .orderBy(F.desc("Ventas_Globales"))
      .limit(10)
)
display(ventas_publicadora)

Publisher,Cantidad_Juegos,Ventas_Globales
Nintendo,696,1784.43
Electronic Arts,1339,1093.39
Activision,966,721.41
Sony Computer Entertainment,682,607.28
Ubisoft,918,473.54
Take-Two Interactive,412,399.3
THQ,712,340.44
Konami Digital Entertainment,823,278.56
Sega,632,270.7
Namco Bandai Games,928,253.65
